In [1]:
# Cell 1 — imports
import torch
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from data.h36m_dataset import build_dataloaders
from utils.trainer import (
    evaluate_model, print_results,
    measure_inference_time, ResultsTracker,
    train_model
)

device = torch.device('cuda' if torch.cuda.is_available()
                      else 'cpu')
print(f"Device: {device}")

DATA_PATH = "D:/L4S2/Research Project in AI/Research/iccv21_git_src/data_3d_h36m.npz"
_, test_loader = build_dataloaders(
    DATA_PATH, batch_size=32,
    train_stride=5, test_stride=1 
)

Device: cpu


d:\Projects\SP_GaRT\SPaRTA-Spatially-Pruned-Affordance-aware-Transformer-Architecture\.venv\Lib\site-packages\numpy\lib\_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


[H36MDataset] 38045 windows | subjects=['S1', 'S5', 'S6', 'S7', 'S8'] | t_obs=10 t_pred=25 | stride=5 | 25Hz
[H36MDataset] 65927 windows | subjects=['S9', 'S11'] | t_obs=10 t_pred=25 | stride=1 | 25Hz


In [2]:
# Cell 2 — zero-velocity baseline as dummy model
class ZeroVelocityBaseline(torch.nn.Module):
    def forward(self, observed):
        last = observed[:, -1:, :, :]
        return last.repeat(1, 25, 1, 1)

zv_model = ZeroVelocityBaseline().to(device)

# Quick evaluation — 10 batches only
zv_results = evaluate_model(
    zv_model, test_loader, device, n_batches=10
)
print_results(zv_results, "Zero-Velocity Baseline")

# Expected:
# MPJPE@560ms  ≈ 226mm
# GVR          ≈ 0.00 (if batch is standing)
# BLE          ≈ 0.00mm


  Zero-Velocity Baseline
  MPJPE (mm) at horizons:
        80ms :    17.16 mm
       160ms :    32.84 mm
       320ms :    58.22 mm
       560ms :    84.28 mm
      1000ms :   116.84 mm
  ADE :    74.57 mm
  FDE :   116.84 mm
  GVR :   0.0000
  BLE :     0.00 mm


d:\Projects\SP_GaRT\SPaRTA-Spatially-Pruned-Affordance-aware-Transformer-Architecture\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [3]:
# Cell 3 — inference speed
mean_ms, std_ms = measure_inference_time(zv_model, device)
print(f"Zero-velocity: {mean_ms:.3f} ± {std_ms:.3f} ms")
# Expected: < 1ms (it is just a tensor repeat operation)

Zero-velocity: 0.080 ± 0.032 ms


In [4]:
# Cell 4 — results tracker
tracker = ResultsTracker()
tracker.add('Zero-Velocity', zv_results, ms=mean_ms)
tracker.print_table()

# Save to results folder
tracker.save('../results/results_table.csv')

  Recorded results for: Zero-Velocity

Model                         80ms  160ms  320ms  560ms  1000ms      ADE      FDE      GVR      BLE        ms
Zero-Velocity                  17.2   32.8   58.2   84.3  116.8     74.6    116.8   0.0000     0.00      0.08
Results saved to: ../results/results_table.csv


In [5]:
# Cell 5 — train_model smoke test
# Train zero-velocity model for 2 epochs to confirm
# the training loop runs without errors
# (Zero-velocity has no parameters so loss will not change
#  — that is expected and fine for this test)

import torch.nn as nn

class TinyLinear(torch.nn.Module):
    """Minimal trainable model for smoke testing."""
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10 * 17 * 3, 25 * 17 * 3)

    def forward(self, obs):
        B = obs.shape[0]
        x = obs.reshape(B, -1)
        out = self.fc(x)
        return out.reshape(B, 25, 17, 3)

train_loader, _ = build_dataloaders(
    DATA_PATH, batch_size=32,
    train_stride=5, test_stride=1
)

tiny_model = TinyLinear().to(device)
train_model(
    model=tiny_model,
    train_loader=train_loader,
    test_loader=test_loader,
    n_epochs=2,
    lr=1e-3,
    device=device,
    experiment_name='smoke_test',
    eval_every=1,
    save_dir='../checkpoints',
    log_dir='../runs',
    resume=False
)
# Expected: 2 epochs run, loss printed each epoch,
# checkpoint saved, no errors

[H36MDataset] 38045 windows | subjects=['S1', 'S5', 'S6', 'S7', 'S8'] | t_obs=10 t_pred=25 | stride=5 | 25Hz
[H36MDataset] 65927 windows | subjects=['S9', 'S11'] | t_obs=10 t_pred=25 | stride=1 | 25Hz
Starting: smoke_test


d:\Projects\SP_GaRT\SPaRTA-Spatially-Pruned-Affordance-aware-Transformer-Architecture\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch   1/2 | loss: 0.01694 | MPJPE@560ms: 143.7mm | GVR: 0.0010249999817460776
  ✓ Best saved: MPJPE@560ms=143.7mm
Epoch   2/2 | loss: 0.00980 | MPJPE@560ms: 133.8mm | GVR: 0.0017749999929219484
  ✓ Best saved: MPJPE@560ms=133.8mm

Done. Best MPJPE@560ms: 133.8mm
Best checkpoint: ../checkpoints/smoke_test_best.pth


TinyLinear(
  (fc): Linear(in_features=510, out_features=1275, bias=True)
)